<a href="https://colab.research.google.com/github/DunyaVatandasi/abd-borsa-calismalari/blob/main/03_temel_analiz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q yfinance
import yfinance as yf
import pandas as pd

def temel_analiz(sembol):
    try:
        bilgi = yf.Ticker(sembol).info
    except Exception as e:
        print(f"Yahoo'dan veri alınamadı: {e}")
        return None

    def yuzde(x): return f"%{x*100:.1f}" if x is not None else "-"
    def sayi(x): return f"{x:.1f}" if x is not None else "-"
    def milyar(x): return f"{x/1e9:,.1f} milyar $" if x is not None else "-"
    g = bilgi.get

    print(f"===== {g('longName', sembol)} ({sembol}) =====")
    print(f"Sektör             : {g('sector', '-')} / {g('industry', '-')}")
    print(f"Piyasa değeri      : {milyar(g('marketCap'))}")
    print("\n--- Değerleme: Pahalı mı, ucuz mu? ---")
    print(f"F/K (son 12 ay)    : {sayi(g('trailingPE'))}")
    print(f"F/K (ileriye dönük): {sayi(g('forwardPE'))}")
    print(f"PD/DD              : {sayi(g('priceToBook'))}")
    print("\n--- Büyüme: Şirket büyüyor mu? ---")
    print(f"Gelir büyümesi     : {yuzde(g('revenueGrowth'))}")
    print(f"Kâr büyümesi       : {yuzde(g('earningsGrowth'))}")
    print("\n--- Kârlılık: Ne kadar verimli? ---")
    print(f"Brüt marj          : {yuzde(g('grossMargins'))}")
    print(f"Faaliyet marjı     : {yuzde(g('operatingMargins'))}")
    print(f"Net kâr marjı      : {yuzde(g('profitMargins'))}")
    print(f"Özkaynak kârlılığı : {yuzde(g('returnOnEquity'))}")
    print("\n--- Finansal sağlık: Sağlam mı? ---")
    print(f"Nakit              : {milyar(g('totalCash'))}")
    print(f"Toplam borç        : {milyar(g('totalDebt'))}")
    print(f"Serbest nakit akışı: {milyar(g('freeCashflow'))}")
    return bilgi

aapl = temel_analiz("AAPL")

===== Apple Inc. (AAPL) =====
Sektör             : Technology / Consumer Electronics
Piyasa değeri      : 5,177.4 milyar $

--- Değerleme: Pahalı mı, ucuz mu? ---
F/K (son 12 ay)    : 39.7
F/K (ileriye dönük): 33.7
PD/DD              : 46.2

--- Büyüme: Şirket büyüyor mu? ---
Gelir büyümesi     : %11.7
Kâr büyümesi       : %14.1

--- Kârlılık: Ne kadar verimli? ---
Brüt marj          : %48.8
Faaliyet marjı     : %33.0
Net kâr marjı      : %27.6
Özkaynak kârlılığı : %142.3

--- Finansal sağlık: Sağlam mı? ---
Nakit              : 56.7 milyar $
Toplam borç        : 94.5 milyar $
Serbest nakit akışı: 105.7 milyar $


In [2]:
def rakip_karsilastir(semboller):
    alanlar = {
        "F/K": "trailingPE", "İleri F/K": "forwardPE",
        "Gelir büy. %": "revenueGrowth", "Net marj %": "profitMargins",
        "ROE %": "returnOnEquity",
    }
    satirlar = {}
    for s in semboller:
        try:
            bilgi = yf.Ticker(s).info
        except Exception:
            print(f"{s} için veri alınamadı, atlanıyor.")
            continue
        satir = {}
        for ad, anahtar in alanlar.items():
            deger = bilgi.get(anahtar)
            if deger is not None and "%" in ad:
                deger *= 100
            satir[ad] = deger
        satirlar[s] = satir
    return pd.DataFrame(satirlar).T.round(1)

rakip_karsilastir(["AAPL", "MSFT", "GOOGL", "AMZN", "META"])

,F/K,İleri F/K,Gelir büy. %,Net marj %,ROE %
AAPL,39.7,33.7,11.7,27.6,142.3
MSFT,27.3,21.7,18.8,41.2,34.2
GOOGL,15.4,19.2,24.7,50.0,44.3
AMZN,21.6,24.4,12.1,16.9,27.4
META,21.9,18.0,26.1,31.1,32.3
